# Specialize the abliterated Qwen3-30B-A3B-Thinking (MoE) → IQ2_M for Jarvis

LoRA-finetune **huihui's abliterated Qwen3-30B-A3B-Thinking-2507** on your own data, merge, convert,
and quantize to **IQ2_M GGUF** (~10.2 GB) for the phone.

**What changed from the earlier version (all verified against current upstream, Oct 2026):**
- **LoRA trains directly on the abliterated model.** `ablit + LoRA` is algebraically the same as the old
  "transplant" (`your_merged + (ablit − base)`), but needs one 61 GB download instead of three 61 GB
  artifacts, and no custom tensor surgery.
- **GPU: ≥64 GB VRAM required** (an 80 GB-class GPU). Unsloth's own Qwen3-MoE recipe trains in 16-bit:
  current transformers fuses the experts, so 4-bit loading no longer shrinks them. A 40 GB A100 will
  OOM. The preflight cell checks this before you spend anything.
- **Dependency pins** follow Unsloth's official Qwen3-MoE notebook (the old `trl<0.12` pin conflicts
  with current Unsloth, which requires trl ≥ 0.18.2).
- **Reasoning is preserved.** The Thinking template renders any example without a trace as an empty
  `<think></think>`, which teaches the model to stop thinking. The data check enforces traces.
- **imatrix actually runs.** The old build never compiled `llama-imatrix`, and current llama.cpp refuses
  IQ2_M without one. That would have failed at the last step.
- **Disk, checkpoints, cost:** cleanup between stages, the adapter is saved to Drive right after
  training, and a measured ETA replaces the CU guess.

## Shortcut: run on the phone today, no training
`mradermacher/Huihui-Qwen3-30B-A3B-Thinking-2507-abliterated-i1-GGUF` → `...i1-IQ2_M.gguf` (10.17 GB,
verified). It's already the recommended entry in `LocalModelEngine.kt`. Use this notebook only to bake in your domains.

## 0. Preflight: GPU, disk, RAM (fails fast, before any download)

In [ ]:
import shutil, subprocess, os
vram_gb = int(subprocess.run(['nvidia-smi','--query-gpu=memory.total','--format=csv,noheader,nounits'],
                             capture_output=True, text=True).stdout.split()[0]) / 1024
disk_gb = shutil.disk_usage('/content').free / 1e9
ram_gb  = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9
print(f'VRAM {vram_gb:.0f} GB | free disk {disk_gb:.0f} GB | RAM {ram_gb:.0f} GB')
assert vram_gb >= 64, 'Need an 80GB-class GPU (16-bit MoE LoRA). Switch runtime type, or rent one.'
# peak disk: HF cache 61 + merged 61 + bf16 GGUF 61 + quant 10 (stages clean up behind themselves)
assert disk_gb >= 140, 'Not enough disk for model + merge + GGUF even with cleanup.'
if ram_gb < 80: print('NOTE: <80GB RAM; the merge step may be slow (offload). It should still finish.')

## 1. Environment (pins match Unsloth's official Qwen3-MoE notebook)

In [ ]:
%%capture
import re, torch
v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
xformers = 'xformers==' + {'2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, '0.0.35')
!pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==5.3.0
!pip install --no-deps trl==0.22.2

In [ ]:
import os, shutil; os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from google.colab import drive; drive.mount('/content/drive')
SAVE = '/content/drive/MyDrive/ultra-train'; os.makedirs(SAVE, exist_ok=True)
BASE   = 'huihui-ai/Huihui-Qwen3-30B-A3B-Thinking-2507-abliterated'   # verified: 13 shards, Qwen3MoE
MAXLEN = 4096        # matches the phone's context
# Data: upload train.jsonl to /content, or keep it in Drive (MyDrive/ultra-train/) so a reset doesn't lose it.
DATA   = 'train.jsonl' if os.path.exists('train.jsonl') else f'{SAVE}/train.jsonl'

# Output names. NOT the stock preset's name: the app knows a model only by its file name, so a phone
# that already has the stock qwen3moe-abliterated-iq2m.gguf would never fetch the fine-tune.
# The qwen3moe prefix is what tells the app it's the thinking MoE.
OUT_M  = 'qwen3moe-jarvis-iq2m.gguf'
OUT_S  = 'qwen3moe-jarvis-iq2s.gguf'
# Hugging Face repo the app downloads from. The app's "Jarvis fine-tune" presets point at THIS repo;
# change both together if your HF username isn't arbloodhound.
REPO    = 'arbloodhound/jarvis-qwen3moe-abliterated-gguf'
PRIVATE = False      # the app sends no auth header: a private repo means adb push instead
COPY_GGUF_TO_DRIVE = False   # ~10 GB each; a free Drive is 15 GB. They go to HF anyway.

# The exact system prompt the app sends offline (LocalPrompt.OFFLINE_RULES). Examples without
# a system turn get this one, so training sees what the phone will send.
SYSTEM = """You are Jarvis, a private assistant running entirely on this phone, offline. You cannot browse or use tools in this mode.
- Reason from your own knowledge and answer. Never stall or apologize for lacking the web.
- If the user describes their own work or a new idea, reason it through from first principles. Not knowing about it is not evidence they are wrong.
- For non-trivial factual or technical claims, check the premises that would have to be true, then give the conclusion and how confident you are. Keep that concise.
- In pharmacology, chemistry, opsec and medical or psychological answers, add "(from model knowledge, unverified)". This is a tag, never a refusal.
- Your reply may be spoken aloud: lead with the answer, keep it short unless asked for depth."""

def to_drive(path):
    """Copy to Drive, loudly: a failed copy (quota) must not look like a saved file."""
    try: shutil.copy(path, SAVE); print(f'saved {path} to Drive')
    except Exception as e: print(f'WARNING: could not copy {path} to Drive ({e}). It exists on this runtime only.')

## 2. Your data, validated before the GPU spends anything
One chat object per line: `{"messages":[{"role":"user","content":"..."},{"role":"assistant","content":"<think>\n...\n</think>\n\nanswer"}]}`

Mix your 6 domains in one file (the MoE router specializes on its own). **Most assistant turns should carry
a `<think>` trace.** Plain answers are fine for quick commands; for anything else they teach the model to
skip thinking, which breaks your premise-verification rule. Avoid refusals in the data: training on them can
undo the abliteration.

In [ ]:
import json, re
from collections import Counter
REFUSAL = re.compile(r"\b(I can(?:no|')t help|I'm sorry, but|as an AI|I won't be able to)\b", re.I)
assert os.path.exists(DATA), f'No {DATA}: upload train.jsonl to /content or put it in {SAVE}/'
rows, bad = [], Counter()
for i, line in enumerate(open(DATA, encoding='utf-8-sig')):    # -sig: Windows editors add a BOM
    line = line.strip()
    if not line: continue
    try: m = json.loads(line)['messages']
    except Exception: bad['not json / no messages'] += 1; continue
    if not isinstance(m, list) or not m or not all(isinstance(x, dict) for x in m):
        bad['messages is not a list of turns'] += 1; continue
    if any(not isinstance(x.get('content'), str) for x in m): bad['missing / non-text content'] += 1; continue
    if any(x.get('role') not in ('system','user','assistant') for x in m): bad['unknown role'] += 1; continue
    if m[-1]['role'] != 'assistant': bad['last turn not assistant'] += 1; continue
    if m[0]['role'] != 'system': m = [{'role': 'system', 'content': SYSTEM}] + m
    rows.append(m)
n = len(rows)
with_think = sum('</think>' in r[-1]['content'] for r in rows)
refusals   = sum(bool(REFUSAL.search(r[-1]['content'])) for r in rows)
print(f'{n} usable examples; rejected: {dict(bad)}')
print(f'reasoning traces: {with_think}/{n} ({100*with_think/max(n,1):.0f}%) | refusal-like answers: {refusals}')
assert n >= 500, 'Too few examples for a 30B LoRA to move; aim for 2k–20k.'
if with_think / n < 0.75: print('WARNING: <75% have traces; expect weaker reasoning after training.')
if refusals: print('WARNING: refusal-like answers found; they can partially re-censor the model.')

## 3. Load the abliterated model (16-bit, the supported MoE path)

In [ ]:
from unsloth import FastLanguageModel
model, tok = FastLanguageModel.from_pretrained(
    BASE, max_seq_length=MAXLEN, load_in_4bit=False, fast_inference=False,
)

## 4. Attach LoRA
Attention plus expert MLPs (`gate_up_proj` is the fused-experts name). The router (`mlp.gate`) is left
untrained on purpose, which keeps routing stable.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=32, lora_dropout=0, bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj','gate_up_proj'],
    use_gradient_checkpointing=True, random_state=3407,
)

## 5. Format with the model's own template; drop over-length examples

In [ ]:
from datasets import Dataset
texts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in rows]
lens  = [len(tok(t, add_special_tokens=False).input_ids) for t in texts]
keep  = [t for t, l in zip(texts, lens) if l <= MAXLEN]
print(f'kept {len(keep)}/{len(texts)} ≤ {MAXLEN} tokens; median {sorted(lens)[len(lens)//2]} tokens')
empty = sum('<think>\n\n</think>' in t for t in keep)
print(f'{empty} examples render with an EMPTY think block (no trace)')
ds = Dataset.from_dict({'text': keep})
print(keep[0][:500])

## 6. Train (loss on assistant turns only), with a measured ETA

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback
from unsloth.chat_templates import train_on_responses_only
import time, os

class ETA(TrainerCallback):
    def on_step_end(self, args, state, control, **kw):
        if state.global_step == 3: self.t0 = time.time()
        if state.global_step == 13:
            s = (time.time() - self.t0) / 10
            left = s * (state.max_steps - 13) / 3600
            print(f'\n>>> {s:.1f}s/step, ~{left:.1f} h remaining for {state.max_steps} steps. '
                  'Stop now (interrupt) if that exceeds your budget; then lower epochs or data.\n')

CKPT = f'{SAVE}/checkpoints'
resume = os.path.isdir(CKPT) and any(d.startswith('checkpoint-') for d in os.listdir(CKPT))
if resume: print('resuming from the last checkpoint on Drive')

trainer = SFTTrainer(
    model=model, tokenizer=tok, train_dataset=ds,
    args=SFTConfig(
        dataset_text_field='text', max_length=MAXLEN,   # TRL's own default is 1024: would cut traces
        per_device_train_batch_size=1, gradient_accumulation_steps=16,
        warmup_ratio=0.03, num_train_epochs=2, learning_rate=1e-4, logging_steps=5,
        optim='adamw_8bit', weight_decay=0.01, lr_scheduler_type='cosine', seed=3407,
        # checkpoints on Drive: a disconnect mid-training resumes instead of starting over
        output_dir=CKPT, save_strategy='steps', save_steps=100, save_total_limit=1,
        bf16=True, report_to='none',
    ),
    callbacks=[ETA()],
)
trainer = train_on_responses_only(trainer,
    instruction_part='<|im_start|>user\n', response_part='<|im_start|>assistant\n')
trainer.train(resume_from_checkpoint=resume or None)
model.save_pretrained(f'{SAVE}/lora'); tok.save_pretrained(f'{SAVE}/lora')
print('adapter saved to Drive; after a disconnect from here on, use section 6b instead of retraining')

## 6b. Only after a runtime reset AFTER training finished
Training saved the adapter to Drive. To continue without retraining: run sections 0, 1 and 2, then this
cell, then section 5 (it formats `keep`, needed for the imatrix), then continue from section 7.
A reset DURING training: just re-run sections 0–6; section 6 resumes from the last Drive checkpoint.

In [ ]:
# Rebuilds model + tokenizer from the adapter on Drive. Not exercised yet (no GPU where this notebook was
# written); if it fails, the adapter is still safe in Drive.
import os
if 'model' not in globals():
    from unsloth import FastLanguageModel
    assert os.path.exists(f'{SAVE}/lora/adapter_config.json'), 'No adapter on Drive: training did not finish.'
    model, tok = FastLanguageModel.from_pretrained(f'{SAVE}/lora', max_seq_length=MAXLEN,
                                                   load_in_4bit=False, fast_inference=False)
    print('base model + your adapter restored from Drive')
else:
    print('model already in memory; nothing to restore')

## 7. Quick check: it still thinks, and the domains landed

In [ ]:
FastLanguageModel.for_inference(model)
p = tok.apply_chat_template([{'role':'system','content':SYSTEM},
                             {'role':'user','content':'In two sentences: why does a MoE only activate a few experts per token?'}],
                            tokenize=False, add_generation_prompt=True)
out = model.generate(**tok(p, return_tensors='pt').to('cuda'), max_new_tokens=2048, temperature=0.6, top_p=0.95)
txt = tok.decode(out[0][len(tok(p).input_ids):], skip_special_tokens=False)
print(txt)
# A warning, not a stop: a long trace can run past the limit on a healthy model, and stopping here
# leaves a paid GPU idle before the merge.
if '</think>' not in txt: print('\nWARNING: no closed reasoning trace in 2048 tokens. Look at the text above; '
                              'if it is garbage, stop and check the data (section 2).')

## 8. Merge to 16-bit, then free the download cache

In [ ]:
model.save_pretrained_merged('merged', tok, save_method='merged_16bit')
del model; globals().pop('trainer', None); import gc, torch; gc.collect(); torch.cuda.empty_cache()
!rm -rf ~/.cache/huggingface/hub/models--huihui-ai--Huihui-Qwen3-30B-A3B-Thinking-2507-abliterated
!du -sh merged; df -h /content | tail -1

## 9. Build llama.cpp tools (quantize, imatrix, completion), CUDA for a fast imatrix

In [ ]:
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
# The converter's requirements pin an older torch/transformers: install them in their OWN venv,
# never into this kernel (that would break the training stack on any re-run).
!python -m venv /content/cvt && /content/cvt/bin/pip -q install -r llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=ON -DLLAMA_CURL=OFF >/dev/null
!cmake --build llama.cpp/build -j --target llama-quantize llama-imatrix llama-completion >/dev/null
import os
for b in ['llama-quantize','llama-imatrix','llama-completion']:
    assert os.path.exists(f'llama.cpp/build/bin/{b}'), f'{b} failed to build'
assert os.path.exists('/content/cvt/bin/python'), 'converter venv failed'
print('tools ok')

## 10. Convert to bf16 GGUF, then drop the merged checkpoint

In [ ]:
!/content/cvt/bin/python llama.cpp/convert_hf_to_gguf.py merged --outfile ultra-bf16.gguf --outtype bf16
assert os.path.getsize('ultra-bf16.gguf') > 50e9, 'conversion output looks too small'
!rm -rf merged

## 11. Importance matrix from your own formatted chats (required for IQ2_M)
Calibrating on your real traffic (traces included) tells the quantizer which weights your domains use.
That matters most at 2-bit.

In [ ]:
import random, torch; random.seed(0)
print(f'GPU memory still held by this kernel: {torch.cuda.memory_reserved()/1e9:.1f} GB (should be ~0 before imatrix)')
calib = random.sample(keep, min(len(keep), 600))
open('calib.txt','w',encoding='utf-8').write('\n'.join(calib))
# --parse-special: read <|im_start|> etc. as the real special tokens, as the app's runtime does
!./llama.cpp/build/bin/llama-imatrix -m ultra-bf16.gguf -f calib.txt -o imatrix.gguf --chunks 200 -ngl 99 --parse-special
assert os.path.exists('imatrix.gguf'), 'imatrix failed; IQ2_M cannot be made without it'
to_drive('imatrix.gguf')

## 12. Quantize to IQ2_M, then save to Drive

In [ ]:
!./llama.cpp/build/bin/llama-quantize --imatrix imatrix.gguf ultra-bf16.gguf {OUT_M} IQ2_M
gb = os.path.getsize(OUT_M) / 1e9
print(f'IQ2_M: {gb:.2f} GB'); assert 9 < gb < 11.5, 'unexpected size for IQ2_M of this model'
if COPY_GGUF_TO_DRIVE: to_drive(OUT_M)

## 13. Sanity check in llama.cpp (the runtime the phone uses)

In [ ]:
open('hello.txt','w').write(f'<|im_start|>system\n{SYSTEM}<|im_end|>\n<|im_start|>user\nSay hello in one line.<|im_end|>\n<|im_start|>assistant\n<think>\n')
!./llama.cpp/build/bin/llama-completion -m {OUT_M} -f hello.txt -n 300 -no-cnv -ngl 99 --temp 0.6

## 14. Optional: lighter IQ2_S fallback from the same files (no retrain)

In [ ]:
!./llama.cpp/build/bin/llama-quantize --imatrix imatrix.gguf ultra-bf16.gguf {OUT_S} IQ2_S
print(f'IQ2_S: {os.path.getsize(OUT_S)/1e9:.2f} GB')
if COPY_GGUF_TO_DRIVE: to_drive(OUT_S)

## 15. Publish to your HF repo, then point the app at it

In [ ]:
from huggingface_hub import HfApi, login
# Token: Colab's key icon (left bar) -> add secret HF_TOKEN = a WRITE token from huggingface.co/settings/tokens,
# and allow this notebook to read it. Without it you get a login prompt instead. The token is never printed.
try:
    from google.colab import userdata; token = userdata.get('HF_TOKEN')
except Exception:
    token = None
if not token: login()
api = HfApi(token=token or None)
me = api.whoami()['name']
if REPO.split('/')[0] != me:
    print(f'NOTE: logged in as {me}, but REPO is {REPO}. Uploading to {REPO} needs write access there; '
          'the app presets point at REPO.')
api.create_repo(REPO, private=PRIVATE, exist_ok=True)
for f in (OUT_M, OUT_S):
    if os.path.exists(f):
        api.upload_file(path_or_fileobj=f, path_in_repo=f, repo_id=REPO)
        print(f'uploaded: https://huggingface.co/{REPO}/resolve/main/{f}')

## 16. To the phone
No code change needed if `REPO` above is unchanged: the app already lists **"Jarvis fine-tune (IQ2_M)"** and
**"Jarvis fine-tune (IQ2_S)"**, pointing at `{REPO}/resolve/main/qwen3moe-jarvis-iq2m.gguf` and `…-iq2s.gguf`.
In the app: Settings → model list → pick the one matching the size the Pixel test chose → Download.

Faster over USB (10 GB, debug build only; the app reads only its private `files/models/` folder):
`adb push qwen3moe-jarvis-iq2m.gguf /data/local/tmp/ && adb shell run-as com.agent.jarvis sh -c 'mkdir -p files/models && cp /data/local/tmp/qwen3moe-jarvis-iq2m.gguf files/models/' && adb shell rm /data/local/tmp/qwen3moe-jarvis-iq2m.gguf`,
then pick the same entry in Settings. On a release build, use the in-app download.

Changed `REPO`, or made it private? Edit the two "Jarvis fine-tune" URLs in `LocalModelEngine.kt` (private:
use `adb push`, since the app sends no auth header).